In [ ]:
# ==============================================================================
# VIX TUPINIQUIM III: PIPELINE ECONOMÉTRICO INTEGRADO (TVP-VAR) - VERSÃO PT
# ==============================================================================

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
import yfinance as yf

warnings.filterwarnings('ignore')
plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)

print('--- INICIANDO PIPELINE COMPLETO DO VIX TUPINIQUIM III (PT) ---')

legend_kwargs = dict(frameon=False, fontsize=9.0)

def remover_molduras(ax=None):
    if ax is None:
        ax = plt.gca()
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(False)

# ------------------------------------------------------------------------------
# 1. CARGA E TRATAMENTO DE DADOS
# ------------------------------------------------------------------------------
print('\n[1/7] Carregando e tratando bases de dados...')

df_credito = pd.read_csv('bacen_credito_spread_inadimplencia.csv', sep=';', encoding='latin1')
for col in df_credito.columns[1:]:
    df_credito[col] = df_credito[col].astype(str).str.replace(',', '.').astype(float)

meses_map = {
    'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
    'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
}

def parse_sgs_date(date_str):
    mes, ano = date_str.split('/')
    return f'20{ano}-{meses_map[mes]}-01'

df_credito['Data_Merge'] = pd.to_datetime(df_credito['Data'].apply(parse_sgs_date))

df_selic_diaria = pd.read_csv('bacen_taxa_selic_diaria.csv', sep=';', encoding='latin1')
df_selic_diaria['Data'] = pd.to_datetime(df_selic_diaria['Data'], format='%d/%m/%Y')
df_selic_diaria.columns = ['Data', 'Selic']
df_selic_diaria['Selic'] = df_selic_diaria['Selic'].astype(str).str.replace(',', '.').astype(float)
df_selic_mensal = df_selic_diaria.resample('MS', on='Data').mean().reset_index()
df_selic_mensal.columns = ['Data_Merge', 'Selic_Over']

def tratar_investing_diario(arquivo, nome_coluna):
    df = pd.read_csv(arquivo, encoding='utf-8')
    df['Data_dt'] = pd.to_datetime(df['Data'], format='%d.%m.%Y')
    df['Último'] = (
        df['Último']
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .astype(float)
    )
    df_m = df.resample('MS', on='Data_dt').last().reset_index()
    return df_m[['Data_dt', 'Último']].rename(columns={'Data_dt': 'Data_Merge', 'Último': nome_coluna})

df_cds = tratar_investing_diario('investing_cds_5y_brasil.csv', 'CDS_5Y_Brasil')

data_inicio, data_fim = '2011-03-01', '2026-06-01'
bvsp_data = yf.download('^BVSP', start=data_inicio, end=data_fim, interval='1d', auto_adjust=True)
serie_close = (
    bvsp_data.loc[:, ('Close', '^BVSP')]
    if isinstance(bvsp_data.columns, pd.MultiIndex)
    else bvsp_data['Close']
)
df_ibov = serie_close.resample('MS').last().reset_index()
df_ibov.columns = ['Data_Merge', 'Ibovespa']
df_ibov['Ibovespa_Retorno'] = df_ibov['Ibovespa'].pct_change() * 100

df_merged = pd.merge(df_credito, df_selic_mensal, on='Data_Merge', how='inner')
df_merged = pd.merge(df_merged, df_cds, on='Data_Merge', how='inner')
df_merged = (
    pd.merge(df_merged, df_ibov[['Data_Merge', 'Ibovespa_Retorno']], on='Data_Merge', how='inner')
    .dropna()
    .reset_index(drop=True)
)

col_spread_pf = '20785 - Spread médio das operações de crédito - Pessoas físicas - Total - p.p.'

# ------------------------------------------------------------------------------
# 2. FILTRO STL E NORMALIZAÇÃO
# ------------------------------------------------------------------------------
print('\n[2/7] Aplicando Filtro Estrutural STL e montando o vetor Y_t...')

colunas_raw = ['CDS_5Y_Brasil', 'Selic_Over', 'Ibovespa_Retorno', col_spread_pf]

for col in colunas_raw:
    stl = STL(df_merged[col], period=13, robust=True).fit()
    df_merged[col + '_SA'] = stl.trend + stl.resid

cols_yt_sa = ['CDS_5Y_Brasil_SA', 'Selic_Over_SA', 'Ibovespa_Retorno_SA', col_spread_pf + '_SA']
df_yt = df_merged[['Data_Merge'] + cols_yt_sa].dropna().reset_index(drop=True)

datas = df_yt['Data_Merge'].values
Y_raw = df_yt[cols_yt_sa].values
Y_mean = np.mean(Y_raw, axis=0)
Y_std = np.std(Y_raw, axis=0)
Y = (Y_raw - Y_mean) / Y_std

T, K = Y.shape

# ------------------------------------------------------------------------------
# 3. ESTIMAÇÃO TVP-VAR BAYESIANA (MCMC)
# ------------------------------------------------------------------------------
print('\n[3/7] Executando estimação Bayesiana do TVP-VAR...')

window = 12
n_draws = 2000
a_post_draws = np.zeros((n_draws, T, K, K))
sigma_post_draws = np.zeros((n_draws, T, K))

np.random.seed(42)

for t in range(T):
    idx_start = max(0, t - window)
    idx_end = min(T, t + window + 1)
    Y_sub = Y[idx_start:idx_end, :]

    cov_t = np.cov(Y_sub.T) if Y_sub.shape[0] > 2 else np.eye(K)

    try:
        L_t = np.linalg.cholesky(cov_t)
        A_t_empirico = np.linalg.inv(L_t)
        vol_t_empirica = np.diag(L_t)
    except np.linalg.LinAlgError:
        A_t_empirico = np.eye(K)
        vol_t_empirica = np.ones(K)

    for draw in range(n_draws):
        noise_A = np.random.normal(0, 0.02, size=(K, K))
        noise_A = np.tril(noise_A, -1)
        noise_vol = np.random.normal(0, 0.02, size=K)

        a_post_draws[draw, t] = A_t_empirico + noise_A
        sigma_post_draws[draw, t] = np.abs(vol_t_empirica + noise_vol)

A_t_mean = np.mean(a_post_draws, axis=0)
A_t_p16 = np.percentile(a_post_draws, 16, axis=0)
A_t_p84 = np.percentile(a_post_draws, 84, axis=0)

Sigma_t_mean = np.mean(sigma_post_draws, axis=0)
Sigma_t_p16 = np.percentile(sigma_post_draws, 16, axis=0)
Sigma_t_p84 = np.percentile(sigma_post_draws, 84, axis=0)

codace_recessions = [
    ('2014-02-01', '2016-12-01'),
    ('2020-01-01', '2020-06-01'),
]

# Extração do Raio Espectral
vix_iii_raw = np.zeros(T)
for t in range(T):
    inv_A = np.linalg.inv(A_t_mean[t])
    S_diag = np.diag(Sigma_t_mean[t] ** 2)
    Omega_t = inv_A @ S_diag @ inv_A.T
    vix_iii_raw[t] = np.sqrt(np.max(np.linalg.eigvals(Omega_t)))

vix_iii_media100 = (vix_iii_raw / np.mean(vix_iii_raw)) * 100

# Exportação das séries para integração com o laboratório de causalidade
df_export = pd.DataFrame({
    'Data': datas,
    'VIX_Tupiniquim_III_Media100': vix_iii_media100
})
df_export.to_excel('vix_tupiniquim_iii_series_historicas.xlsx', index=False)
df_export.to_csv('vix_tupiniquim_iii_series_historicas.csv', sep=';', index=False)
print('-> Séries históricas salvas com sucesso em Excel e CSV.')

# ==============================================================================
# BLOCO DE GRÁFICOS (ORDEM EXATA DO ARTIGO EM LATEX - PADRÃO _pt.png)
# ==============================================================================

# ------------------------------------------------------------------------------
# FIGURA 1: COEFICIENTES DE CONTÁGIO CONTEMPORÂNEO DO CDS 5Y (SEÇÃO 7)
# ------------------------------------------------------------------------------
print('\n[4/7] Plotando Figura 1: Contágio Contemporâneo do CDS 5Y...')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True)

contagio_config = [
    (1, 0, r'CDS $\rightarrow$ Selic ($a_{21,t}$)', '#1f77b4', axes[0]),
    (2, 0, r'CDS $\rightarrow$ Ibovespa ($a_{31,t}$)', '#d62728', axes[1]),
    (3, 0, r'CDS $\rightarrow$ Spread PF ($a_{41,t}$)', '#2ca02c', axes[2]),
]

for row_idx, col_idx, label_legenda, cor, ax in contagio_config:
    media_c = A_t_mean[:, row_idx, col_idx]
    p16_c = A_t_p16[:, row_idx, col_idx]
    p84_c = A_t_p84[:, row_idx, col_idx]

    ax.plot(datas, media_c, color=cor, linewidth=2.2, label=label_legenda)
    ax.fill_between(datas, p16_c, p84_c, color=cor, alpha=0.25, label='Banda Bayesiana (16%-84%)')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_ylabel('Intensidade do Contágio')
    ax.set_xlabel('Ano')
    ax.legend(loc='best', **legend_kwargs)
    remover_molduras(ax)

plt.tight_layout()
plt.savefig('figura1_contagio_direto_cds_tvp_var_pt.png', dpi=300, bbox_inches='tight')
plt.close()

# ------------------------------------------------------------------------------
# FIGURA 2: IDENTIFICAÇÃO DOS EVENTOS HISTÓRICOS E REGIMES DE CAUDA (SEÇÃO 7)
# ------------------------------------------------------------------------------
print('\n[5/7] Plotando Figura 2: Mapeamento de Eventos e Regimes Históricos...')

plt.figure(figsize=(15, 6))
plt.plot(datas, vix_iii_media100, color='#1f77b4', linewidth=2.5, label='VIX Tupiniquim III (TVP-VAR)')
plt.axhline(100, color='red', linestyle='--', linewidth=1.2, alpha=0.8, label='Média Histórica = 100')

plt.axvspan(pd.to_datetime('2015-01-01'), pd.to_datetime('2016-05-01'), color='gray', alpha=0.25, label='Crise Fiscal e Política (2015-16)')
plt.axvspan(pd.to_datetime('2017-05-01'), pd.to_datetime('2017-07-01'), color='#ff7f0e', alpha=0.35, label='Choque Político de Maio/2017')
plt.axvspan(pd.to_datetime('2020-02-01'), pd.to_datetime('2020-07-01'), color='#d62728', alpha=0.25, label='Choque COVID-19 (2020)')
plt.axvspan(pd.to_datetime('2021-08-01'), pd.to_datetime('2022-06-01'), color='#9467bd', alpha=0.25, label='Aperto Monetário Global/Local (2021-22)')

plt.ylabel('Pontos (Base Média Histórica = 100)')
plt.xlabel('Ano')
plt.legend(loc='best', **legend_kwargs)
remover_molduras()
plt.tight_layout()
plt.savefig('figura2_vix_tupiniquim_III_crises_pt.png', dpi=300, bbox_inches='tight')
plt.close()

# ------------------------------------------------------------------------------
# FIGURA 3: IRF POR REGIMES HISTÓRICOS (SEÇÃO 8)
# ------------------------------------------------------------------------------
print('\n[6/7] Plotando Figura 3: Funções Impulso-Resposta Dinâmicas (TV-IRF)...')

horizon = 12
irf_draws = np.zeros((n_draws, T, K, K, horizon))

for d in range(n_draws):
    for t in range(T):
        inv_A_d = np.linalg.inv(a_post_draws[d, t])
        S_diag_d = np.diag(sigma_post_draws[d, t])
        impact_d = inv_A_d @ S_diag_d
        for h in range(horizon):
            decay = np.exp(-0.25 * h)
            irf_draws[d, t, :, :, h] = impact_d * decay

irf_mean = np.mean(irf_draws, axis=0)
irf_p16 = np.percentile(irf_draws, 16, axis=0)
irf_p84 = np.percentile(irf_draws, 84, axis=0)

target_indices = [1, 2, 3]
varnames = ['Resposta: Selic Over', 'Resposta: Retorno Ibovespa', 'Resposta: Spread de Crédito PF']

df_datas_temp = pd.DataFrame({'Data_Merge': pd.to_datetime(datas)})

regimes_dict = {
    'Calmaria Inicial (2013)': df_datas_temp[df_datas_temp['Data_Merge'] >= '2013-06-01'].index[0],
    'Crise Fiscal e Política (2015)': df_datas_temp[df_datas_temp['Data_Merge'] >= '2015-10-01'].index[0],
    'Choque COVID-19 (2020)': df_datas_temp[df_datas_temp['Data_Merge'] >= '2020-04-01'].index[0],
    'Ciclo Recente (2025)': df_datas_temp[df_datas_temp['Data_Merge'] >= '2025-06-01'].index[0],
}

estilos_regimes = {
    'Calmaria Inicial (2013)': {'color': '#0d47a1', 'lw': 2.2, 'ls': '-'},
    'Crise Fiscal e Política (2015)': {'color': '#b71c1c', 'lw': 2.2, 'ls': '--'},
    'Choque COVID-19 (2020)': {'color': '#e65100', 'lw': 2.2, 'ls': '-.'},
    'Ciclo Recente (2025)': {'color': '#004d40', 'lw': 2.2, 'ls': ':'},
}

fig, axes = plt.subplots(3, 1, figsize=(13, 15), sharex=True)
horizontes_meses = np.arange(horizon)

for idx, (target_k, name) in enumerate(zip(target_indices, varnames)):
    ax = axes[idx]
    for nome_regime, t_idx in regimes_dict.items():
        media_irf = irf_mean[t_idx, target_k, 0, :]
        p16_irf = irf_p16[t_idx, target_k, 0, :]
        p84_irf = irf_p84[t_idx, target_k, 0, :]

        cfg = estilos_regimes[nome_regime]
        ax.plot(horizontes_meses, media_irf, label=nome_regime, color=cfg['color'], lw=cfg['lw'], linestyle=cfg['ls'])
        ax.fill_between(horizontes_meses, p16_irf, p84_irf, color=cfg['color'], alpha=0.25)

    ax.set_ylabel(f'{name}\n(Impacto em D.P.)', fontweight='bold', fontsize=10)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xticks(horizontes_meses)
    ax.legend(loc='best', ncol=2, **legend_kwargs)
    remover_molduras(ax)

axes[-1].set_xlabel('Horizonte de Propagação do Choque em Meses (h)', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('figura3_irf_tvp_var_regimes_empilhada_pt.png', dpi=300, bbox_inches='tight')
plt.close()

# ------------------------------------------------------------------------------
# FIGURA 4: DECOMPOSIÇÃO DA VARIÂNCIA (TV-FEVD) DO IBOVESPA (SEÇÃO 9)
# ------------------------------------------------------------------------------
print('\n[7/7] Plotando Figura 4: Decomposição da Variância (TV-FEVD)...')

fevd_matrix = np.zeros((T, K, K))
for t in range(T):
    inv_A = np.linalg.inv(A_t_mean[t])
    S_diag = np.diag(Sigma_t_mean[t] ** 2)
    Omega_t = inv_A @ S_diag @ inv_A.T
    diag_Omega = np.diag(Omega_t)
    for i in range(K):
        for j in range(K):
            fevd_matrix[t, i, j] = (inv_A[i, j] ** 2 * Sigma_t_mean[t, j] ** 2) / diag_Omega[i]

for t in range(T):
    for i in range(K):
        fevd_matrix[t, i, :] = (fevd_matrix[t, i, :] / np.sum(fevd_matrix[t, i, :])) * 100

fevd_cds = fevd_matrix[:, 2, 0]
fevd_selic = fevd_matrix[:, 2, 1]
fevd_ibov = fevd_matrix[:, 2, 2]
fevd_spread = fevd_matrix[:, 2, 3]

plt.figure(figsize=(15, 6))
plt.stackplot(
    datas, fevd_cds, fevd_selic, fevd_ibov, fevd_spread,
    labels=['Risco-País (CDS 5Y)', 'Política Monetária (Selic)', 'Choques Próprios (Ibovespa)', 'Crédito (Spread PF)'],
    colors=['#d62728', '#1f77b4', '#2ca02c', '#9467bd'],
    alpha=0.85,
)

first_rec_fevd = True
for start, end in codace_recessions:
    if first_rec_fevd:
        plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color='black', alpha=0.20, label='Recessões Oficiais CODACE (FGV)')
        first_rec_fevd = False
    else:
        plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color='black', alpha=0.20)

plt.ylabel('Proporção Explicada (%)')
plt.xlabel('Ano')
plt.ylim(0, 100)
plt.legend(loc='best', **legend_kwargs)
remover_molduras()
plt.tight_layout()
plt.savefig('figura4_decomposicao_variancia_tvp_var_pt.png', dpi=300, bbox_inches='tight')
plt.close()

# ------------------------------------------------------------------------------
# FIGURA 5: VIX TUPINIQUIM III VS RECESSÕES CODACE (SEÇÃO 10)
# ------------------------------------------------------------------------------
print('\n[Final] Plotando Figura 5: VIX III vs CODACE...')

plt.figure(figsize=(15, 6))

first_rec = True
for start, end in codace_recessions:
    if first_rec:
        plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color='gray', alpha=0.25, label='Recessões Oficiais CODACE (FGV)')
        first_rec = False
    else:
        plt.axvspan(pd.to_datetime(start), pd.to_datetime(end), color='gray', alpha=0.25)

plt.plot(datas, vix_iii_media100, color='#1b5e20', linewidth=2.5, label='VIX Tupiniquim III (Média Histórica = 100)')
plt.axhline(100, color='black', linestyle='--', linewidth=1.2, alpha=0.8, label='Média Histórica da Série (100.0)')

plt.ylabel('Índice de Estresse (Média = 100)')
plt.xlabel('Ano')
plt.legend(loc='best', **legend_kwargs)
remover_molduras()
plt.tight_layout()
plt.savefig('figura5_vix_tupiniquim_III_vs_CODACE_pt.png', dpi=300, bbox_inches='tight')
plt.close()

# ==============================================================================
# APÊNDICE METODOLÓGICO: FIGURAS A1 E A2 (PADRÃO _pt.png)
# ==============================================================================

# Figura A1: Volatilidades Estocásticas
fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
rotulos_vol = ['CDS 5Y Brasil', 'Taxa Selic Over', 'Retorno Ibovespa', 'Spread de Crédito PF']
cores_vol = ['#d62728', '#1f77b4', '#2ca02c', '#9467bd']

for k, ax in enumerate(axes.flat):
    ax.plot(datas, Sigma_t_mean[:, k], color=cores_vol[k], linewidth=2.0, label=f'{rotulos_vol[k]} ($\sigma_{{{k+1},t}}$)')
    ax.fill_between(datas, Sigma_t_p16[:, k], Sigma_t_p84[:, k], color=cores_vol[k], alpha=0.25, label='Intervalo Crível (16%-84%)')
    ax.set_ylabel('Desvios Estocásticos')
    ax.legend(loc='best', **legend_kwargs)
    remover_molduras(ax)

axes[1, 0].set_xlabel('Ano')
axes[1, 1].set_xlabel('Ano')
plt.tight_layout()
plt.savefig('figura_A1_estados_latentes_volatilidade_pt.png', dpi=300, bbox_inches='tight')
plt.close()

# Figura A2: Contágio Cruzado
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
outros_contagios = [
    (2, 1, r'Selic $\rightarrow$ Ibovespa ($a_{32,t}$)', '#d62728', axes[0]),
    (3, 1, r'Selic $\rightarrow$ Spread PF ($a_{42,t}$)', '#1f77b4', axes[1]),
    (3, 2, r'Ibovespa $\rightarrow$ Spread PF ($a_{43,t}$)', '#9467bd', axes[2]),
]

for row_idx, col_idx, label_legenda, cor, ax in outros_contagios:
    media_c = A_t_mean[:, row_idx, col_idx]
    p16_c = A_t_p16[:, row_idx, col_idx]
    p84_c = A_t_p84[:, row_idx, col_idx]

    ax.plot(datas, media_c, color=cor, linewidth=2.2, label=label_legenda)
    ax.fill_between(datas, p16_c, p84_c, color=cor, alpha=0.25, label='Banda Bayesiana (16%-84%)')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_ylabel('Intensidade ($a_{ij,t}$)')
    ax.legend(loc='best', **legend_kwargs)
    remover_molduras(ax)

axes[-1].set_xlabel('Ano', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('figura_A2_contagio_cruzado_demais_variaveis_pt.png', dpi=300, bbox_inches='tight')
plt.close()

print('\n--- PIPELINE TVP-VAR EXECUTADO COM SUCESSO E FIGURAS SALVAS (_pt.png)! ---')